# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 5: Fine-tuning a Frontier Model

Now we will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

In [16]:
# imports

import os
import sys
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate
from litellm import completion


sys.path.append(os.path.abspath(os.path.join(os.getcwd(),'..')))
from ai_tools.tools import LLMQuery

In [3]:
# environment

LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
if LITE_MODE:
    username = "Rodan009"
else:
    username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [5]:
openai = OpenAI()

# Data size

OpenAI recommends fine-tuning with a small population of 50-100 examples

I'm going to go with 20,000 points.

This cost me $3.42 - you should stick with 100 examples and the cost will be minimal!

In [6]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)


fine_tune_train = train[:100]
fine_tune_validation = val[:50]

In [7]:
len(fine_tune_train)

100

# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [54]:
def messages_for(item):
    SYSTEM_PROMPT = "Estimate the price of a product based on a description, you **must** only return a price, no explanation."
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item.summary},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [53]:
def gpt_4_nano(item):
    response = completion(model="openai/gpt-4.1-nano-2025-04-14", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [24]:
f"{gpt_4_nano(test[50])} vs. {test[0].price} for {test[0].title}"

'$20.99 vs. 219.0 for Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal'

In [51]:
messages_for(fine_tune_train[0])

[{'role': 'system',
  'content': 'Estimate the price of a product based on a description, you **must** only return a price, no explanation.'},
 {'role': 'user',
  'content': 'Title: Schlage Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half)\nCategory: Door Hardware\nBrand: Schlage\nDescription: Interior half of a two-piece Andover knob with deadbolt in oil rubbed bronze.\nDetails: Includes knob and deadbolt; non-handed knob style; requires F58 to complete handle set; 4" minimum center-to-center door prep; Lifetime Mechanical and Finish Warranty.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [25]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [26]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "system", "content": "Estimate the price of a product based on a description, you **must** only return a price, no explanation."}, {"role": "user", "content": "Title: Schlage Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half)\nCategory: Door Hardware\nBrand: Schlage\nDescription: Interior half of a two-piece Andover knob with deadbolt in oil rubbed bronze.\nDetails: Includes knob and deadbolt; non-handed knob style; requires F58 to complete handle set; 4\" minimum center-to-center door prep; Lifetime Mechanical and Finish Warranty."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "system", "content": "Estimate the price of a product based on a description, you **must** only return a price, no explanation."}, {"role": "user", "content": "Title: KiCA Jetfan 1.0 Mini Electric Air Duster Fan Blower\nCategory: Electronics\nBrand: KiCA\nDescription: A compact, high-speed electric air duster and blower for electronics cleaning, 

In [27]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [28]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [29]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [30]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [31]:
train_file

FileObject(id='file-UJisPkhmNZUtsww2TwFZ5E', bytes=65424, created_at=1769096240, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [32]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [33]:
validation_file

FileObject(id='file-NaSiqhynZGm8DsVB5Th8E9', bytes=32764, created_at=1769096247, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

https://platform.openai.com/storage/files/

# Step 2

## And now time to Fine-tune!

In [34]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

FineTuningJob(id='ftjob-haN0w3W4SMjQu5GWjgNpUnN5', created_at=1769096483, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-VkVx1f1Conr0Yf5jSRPlaQl4', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-UJisPkhmNZUtsww2TwFZ5E', validation_file='file-NaSiqhynZGm8DsVB5Th8E9', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None)

In [ ]:
# fine tuning job using reiforcment training with a grader calculating the difference 
# between the predicted price and the actual price with cleaning the predicted price (via regex)

In [35]:
openai.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-haN0w3W4SMjQu5GWjgNpUnN5', created_at=1769096483, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-VkVx1f1Conr0Yf5jSRPlaQl4', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-UJisPkhmNZUtsww2TwFZ5E', validation_file='file-NaSiqhynZGm8DsVB5Th8E9', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None)], has_more=False, object='list')

In [36]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [37]:
job_id

'ftjob-haN0w3W4SMjQu5GWjgNpUnN5'

In [38]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-haN0w3W4SMjQu5GWjgNpUnN5', created_at=1769096483, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-VkVx1f1Conr0Yf5jSRPlaQl4', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-UJisPkhmNZUtsww2TwFZ5E', validation_file='file-NaSiqhynZGm8DsVB5Th8E9', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None)

In [40]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-3H86Z23o6y0sSw9dwwzvZUS8', created_at=1769096555, level='info', message='Fine-tuning job started', object='fine_tuning.job.event', data=None, type='message'),
 FineTuningJobEvent(id='ftevent-g831UVgQmDxA3IuSX8Bf5fpC', created_at=1769096553, level='info', message='Files validated, moving job to queued state', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-SBZbiASCJEn5wBCZmWbyKrsx', created_at=1769096483, level='info', message='Validating training file: file-UJisPkhmNZUtsww2TwFZ5E and validation file: file-NaSiqhynZGm8DsVB5Th8E9', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-bV4sZ3NI2DrFgONY6B6flwn1', created_at=1769096483, level='info', message='Created fine-tuning job: ftjob-haN0w3W4SMjQu5GWjgNpUnN5', object='fine_tuning.job.event', data={}, type='message')]

https://platform.openai.com/finetune


# Step 3

Test our fine tuned model

In [41]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [42]:
fine_tuned_model_name

'ft:gpt-4.1-nano-2025-04-14:personal:pricer:D0rNNyX9'

In [57]:
# The prompt

def test_messages_for(item):
    SYSTEM_PROMPT = "Estimate the price of a product based on a description, you **must** only return a price, no explanation."
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item.summary}
    ]

In [58]:
# Try this out

test_messages_for(test[0])

[{'role': 'system',
  'content': 'Estimate the price of a product based on a description, you **must** only return a price, no explanation.'},
 {'role': 'user',
  'content': 'Title: Old Blood Noise Endeavors Excess V2 Pedal\nCategory: Electronics\nBrand: Old Blood Noise Endeavors\nDescription: A versatile 3-mode modulation with integrated distortion that delivers delay, chorus, and harmonized fifths in series or parallel paths.\nDetails: Features Delay, Chorus, and Fifth modes with Time/Depth/Volume; Distortion with Gain/Tone/Volume; configurable order (modulation into distortion, distortion into modulation, or parallel); soft-touch switching and true bypass; expression jack for Rate/Depth; internal trimpot for wet-dry mix; requires 125mA 9VDC power.'}]

In [59]:
# The inference function


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [60]:
test_item = test[55]
print(f"real price: {test_item.price}")
print(f"fine tuned: {gpt_4__1_nano_fine_tuned(test_item)}")
print(f"regular: {gpt_4_nano(test_item)}")

real price: 134.12
fine tuned: $148.36
regular: $150


In [61]:
evaluate(gpt_4__1_nano_fine_tuned, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $284 $17 $33 $114 $40 $90 $18 $74 $40 $393 $21 $8 $8 $24 $3 $71 $24 $190 $22 $43 $93 $22 $216 $172 $250 $58 $4 $177 $64 $18 $21 $40 $6 $0 $240 $76 $6 $106 $14 $11 $42 $14 $119 $46 $111 $27 $12 $69 $72 $29 $117 $209 $5 $24 $49 $3 $104 $110 $18 $62 $38 $36 $129 $120 $54 $21 $384 $149 $81 $13 $6 $27 $15 $25 $9 $81 $15 $2 $10 $35 $15 $9 $61 $10 $24 $658 $59 $17 $9 $8 $25 $2 $6 $2 $138 $3 $403 $123 $814 $19 $7 $29 $138 $171 $483 $17 $360 $2 $24 $10 $556 $108 $79 $4 $33 $7 $20 $54 $103 $40 $1780 $20 $9 $148 $52 $4 $40 $23 $71 $914 $13 $124 $4 $163 $2 $35 $62 $19 $102 $23 $119 $41 $5 $421 $178 $24 $1447 $32 $5 $10 $213 $1 $35 $1 $91 $19 $23 $223 $0 $60 $20 $220 $0 $440 $3 $169 $30 $15 $3 $0 $0 $40 $65 $32 $16 $3 $19 $68 $261 $286 $8 $153 $151 $5 $2 $73 $30 $35 $59 $50 $86 $15 $36 $67 $60 $10 $111 $20 $3 

In [63]:
def eval_gpt_4_nano(item):
    response = completion(model="openai/gpt-4.1-nano-2025-04-14", messages=test_messages_for(item), seed=42)
    return response.choices[0].message.content

evaluate(eval_gpt_4_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$19 $34 $10 $25 $45 $155 $98 $65 $9 $20 $363 $20 $25 $21 $19 $8 $41 $0 $40 $69 $59 $56 $65 $25 $82 $273 $45 $5 $251 $64 $55 $10 $10 $40 $35 $269 $15 $31 $34 $13 $80 $35 $20 $5 $70 $0 $27 $13 $70 $52 $20 $105 $225 $0 $197 $16 $18 $50 $52 $3 $106 $3 $46 $10 $221 $30 $90 $295 $25 $74 $7 $18 $130 $6 $35 $21 $126 $5 $8 $3 $30 $0 $15 $69 $2 $60 $62 $56 $0 $21 $3 $10 $5 $10 $4 $78 $11 $193 $20 $225 $20 $3 $12 $11 $49 $32 $10 $350 $19 $151 $20 $136 $44 $8 $54 $79 $5 $5 $294 $147 $19 $511 $50 $84 $50 $90 $30 $51 $29 $19 $89 $13 $35 $5 $85 $0 $85 $50 $53 $42 $11 $249 $5 $4 $6 $38 $30 $690 $85 $17 $4 $144 $2 $210 $6 $29 $71 $41 $30 $25 $61 $13 $33 $2 $390 $3 $502 $25 $5 $5 $0 $3 $20 $12 $57 $101 $3 $57 $24 $57 $296 $10 $150 $201 $100 $13 $73 $83 $30 $7 $0 $99 $25 $11 $41 $70 $20 $70 $6 $19 

In [ ]:
# 96.58 - mini 200
# 79.29 - mini 2000
# 82.26 - nano 2000
# 67.75 - nano 20,000